# Day 28: Multiple Tools

On Day 17 we gave the LLM multiple tools — but we had to write `FunctionDeclaration`, `Schema`, and a dispatcher manually.

Today: three tools, zero boilerplate. LangChain handles the routing.

## Setup

In [20]:
import os, requests
from dotenv import load_dotenv
from langchain.tools import tool
from langchain.agents import create_agent

load_dotenv(dotenv_path='../.env')
os.environ["GOOGLE_API_KEY"] = os.environ["GEMINI_API_KEY2"]

HEADERS = {"User-Agent": "genai-50-days-demo/1.0"}

def get_text(result):
    content = result["messages"][-1].content
    if isinstance(content, list):
        return " ".join(c.get("text", "") for c in content if isinstance(c, dict))
    return content

## Tool 1: Weather

In [21]:
@tool
def get_weather(city: str) -> str:
    """Get current weather for a city."""
    response = requests.get(f"https://wttr.in/{city}?format=3", headers=HEADERS)
    return response.text if response.status_code == 200 else f"Weather unavailable for {city}"

print(get_weather.invoke("Tokyo"))

tokyo: â  +16Â°C



## Tool 2: Wikipedia

In [22]:
@tool
def wikipedia(topic: str) -> str:
    """Look up a topic on Wikipedia and return a summary."""
    slug = topic.strip().replace(" ", "_")
    r = requests.get(f"https://en.wikipedia.org/api/rest_v1/page/summary/{slug}", headers=HEADERS)
    return r.json().get("extract", "No summary found.") if r.status_code == 200 else f"Not found: {topic}"

print(wikipedia.invoke("Eiffel Tower")[:150])

The Eiffel Tower is a lattice tower on the Champ de Mars in Paris, France. It is named after the engineer Gustave Eiffel, whose company designed and b


## Tool 3: Word Definition

In [23]:
@tool
def define_word(word: str) -> str:
    """Get the definition of an English word."""
    r = requests.get(f"https://api.dictionaryapi.dev/api/v2/entries/en/{word}")
    if r.status_code == 200:
        meanings = r.json()[0]["meanings"][0]
        definition = meanings["definitions"][0]["definition"]
        part_of_speech = meanings["partOfSpeech"]
        return f"{word} ({part_of_speech}): {definition}"
    return f"Definition not found for: {word}"

print(define_word.invoke("ephemeral"))

ephemeral (noun): Something which lasts for a short period of time.


## Create the Agent

In [24]:
agent = create_agent(
    "google_genai:gemini-2.5-flash",
    tools=[get_weather, wikipedia, define_word]
)

print("✅ Agent ready with 3 tools")

Both GOOGLE_API_KEY and GEMINI_API_KEY are set. Using GOOGLE_API_KEY.


✅ Agent ready with 3 tools


## Test: Agent Picks the Right Tool

In [25]:
# Should use get_weather
result = agent.invoke({"messages": [{"role": "user", "content": "What's the weather in Paris?"}]})
print("🌤️", get_text(result))

🌤️ The weather in Paris is +12°C.


In [26]:
# Should use wikipedia
result = agent.invoke({"messages": [{"role": "user", "content": "Tell me about the Eiffel Tower"}]})
print("📚", get_text(result))

📚 The Eiffel Tower is a lattice tower on the Champ de Mars in Paris, France. It is named after the engineer Gustave Eiffel, whose company designed and built the tower from 1887 to 1889.


In [27]:
# Should use define_word
result = agent.invoke({"messages": [{"role": "user", "content": "What does ephemeral mean?"}]})
print("📖", get_text(result))

📖 Ephemeral means lasting for a short period of time.


## Key Takeaways

1. Pass a list of `@tool` functions to `create_agent` — the LLM **picks the right one** automatically
2. The docstring is the tool description — **clear docstrings = better tool selection**
3. No dispatcher, no schema, no routing logic — LangChain **replaced 50 lines with a list**